# 27 — Prompt Portability and Multi-Model Systems

    ## Scenario and success criteria

    Two provider adapters must satisfy one output contract; failover is permitted for a read-only summary but not an unconfirmed side effect.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Normalize provider responses at adapters.
- Run the same conformance fixtures across providers.
- Constrain fallback by operation semantics.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 27 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

Blind failover can duplicate writes, change policy behavior, or silently return incompatible output.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab27 import FixtureAdapter, Summary, conformance, route

text = "Acme Corp reports a revenue drop due to supply chain issues."
expected = Summary(company="Acme Corp", revenue_trend="DOWN", risks=("supply chain",))
primary = FixtureAdapter("primary")
fallback = FixtureAdapter("fallback")
outage = FixtureAdapter("primary", available=False)

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
print("primary conformance", conformance(primary, [(text, expected)]))
print("fallback conformance", conformance(fallback, [(text, expected)]))
print("normal route", route(primary, fallback, text, operation_is_read_only=True))
print("outage route", route(outage, fallback, text, operation_is_read_only=True))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert conformance(primary, [(text, expected)]) == (1, 1)
assert route(outage, fallback, text, operation_is_read_only=True).fallback_used
try:
    route(outage, fallback, text, operation_is_read_only=False)
except RuntimeError as error:
    assert "unsafe_fallback" in str(error)
else:
    raise AssertionError("side-effecting failover was allowed")

## Production upgrade

Pin adapter and provider versions, normalize errors and usage, test schemas and safety policy per provider, enforce idempotency for retries, and decide whether degraded mode should abstain rather than silently switch.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.